#  Notebook 02 — Data Cleaning
## Spotify Music Analytics Dataset (2015–2025)

**Goal:** Apply each cleaning function from `src/data_cleaning.py` step-by-step, show before/after comparisons, and save the cleaned dataset to `data/processed/`.

> Every decision is documented below — this notebook is the "audit trail" of our data quality work.


In [ ]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from data_cleaning import (
    load_raw_data, normalize_column_names, remove_duplicates,
    handle_missing_values, fix_data_types, remove_outliers,
    engineer_features, clean_pipeline
)

RAW_PATH = '../data/raw/spotify_2015_2025_85k.csv'
df_raw = load_raw_data(RAW_PATH)
print(f"Raw shape: {df_raw.shape}")


## Step 1 — Normalize Column Names
**Why:** Inconsistent casing (e.g. `Track_Name` vs `track_name`) causes `KeyError` bugs. Standardising to `lowercase_snake_case` makes all downstream code predictable.


In [ ]:
print("BEFORE:", df_raw.columns.tolist())
df_step1 = normalize_column_names(df_raw)
print("AFTER :", df_step1.columns.tolist())


## Step 2 — Remove Duplicates
**Why:** Duplicate rows skew statistics — a song counted twice inflates its stream count and biases popularity averages. We remove full row duplicates here.


In [ ]:
print(f"BEFORE: {len(df_step1):,} rows")
df_step2 = remove_duplicates(df_step1)
print(f"AFTER : {len(df_step2):,} rows")


## Step 3 — Handle Missing Values
**Why:** Most ML models and aggregation functions break on `NaN`. Our strategy:
- Numerical → **median fill** (robust to outliers, unlike mean)
- Categorical → **'Unknown'** (preserves row, makes nulls queryable)
- Rows >50% null → **drop** (too incomplete to be useful)


In [ ]:
print(f"BEFORE — total nulls: {df_step2.isnull().sum().sum()}")
df_step3 = handle_missing_values(df_step2)
print(f"AFTER  — total nulls: {df_step3.isnull().sum().sum()}")


## Step 4 — Fix Data Types
**Why:** `release_date` stored as a string prevents any date-based filtering or year extraction. `mode` and `key` are integers that represent musical categories (not numbers to be averaged). Fixing types saves memory and prevents subtle calculation errors.


In [ ]:
print("BEFORE — release_date dtype:", df_step3['release_date'].dtype)
df_step4 = fix_data_types(df_step3)
print("\nAFTER  — release_date dtype:", df_step4['release_date'].dtype)
print("\nMemory BEFORE:", round(df_step3.memory_usage(deep=True).sum()/1e6, 2), "MB")
print("Memory AFTER :", round(df_step4.memory_usage(deep=True).sum()/1e6, 2), "MB")


## Step 5 — Feature Engineering
**Why:** Raw columns like `duration_ms` are hard to reason about. Derived features like `popularity_tier` and `energy_level` make the data immediately useful for segmentation and visualisation without losing the original values.


In [ ]:
print(f"BEFORE: {df_step4.shape[1]} columns")
df_clean = engineer_features(df_step4)
print(f"AFTER : {df_clean.shape[1]} columns")
print("\nNew columns added:")
new_cols = set(df_clean.columns) - set(df_step4.columns)
for c in sorted(new_cols):
    print(f"  + {c}")
print("\nPopularity tier distribution:")
print(df_clean['popularity_tier'].value_counts().to_string())
print("\nEnergy level distribution:")
print(df_clean['energy_level'].value_counts().to_string())


##  Data Quality Report — Raw vs Cleaned


In [ ]:
report = pd.DataFrame({
    'Metric': ['Rows', 'Columns', 'Missing cells', 'Duplicate rows',
               'Memory (MB)', 'release_date dtype'],
    'Raw': [
        f"{df_raw.shape[0]:,}",
        df_raw.shape[1],
        df_raw.isnull().sum().sum(),
        df_raw.duplicated().sum(),
        f"{df_raw.memory_usage(deep=True).sum()/1e6:.1f}",
        str(df_raw['release_date'].dtype),
    ],
    'Cleaned': [
        f"{df_clean.shape[0]:,}",
        df_clean.shape[1],
        df_clean.isnull().sum().sum(),
        df_clean.duplicated().sum(),
        f"{df_clean.memory_usage(deep=True).sum()/1e6:.1f}",
        str(df_clean['release_date'].dtype),
    ]
})
display(report.set_index('Metric'))

# Save cleaned CSV
os.makedirs('../data/processed', exist_ok=True)
df_clean.to_csv('../data/processed/spotify_cleaned.csv', index=False)
print("\n Cleaned dataset saved → data/processed/spotify_cleaned.csv")
